After processing files of data, here we:

- Merge all processed files (per run and per ldc) by production conditions.
- Tag global events by type of particle (electron or alpha-like) and detector region.
- Store final HDF5 merged file for subsequent analysis.

__NOTE:__ Summary file @ run-level is created.

In [1]:
import sys
sys.path.append('/lhome/ific/c/ccortesp/Analysis/')

from libs import crudo

from datetime import datetime
import glob
import os
import pandas as pd
import numpy as np

%matplotlib inline
%load_ext autoreload
%autoreload 2

# Configuration

In [2]:
# --- NOTEBOOK CONFIGURATION --- #
FILE_TAG = 'LPR_p2_v2'
DATA_PERIOD = 2

In [3]:
# ------------------------------
# DIRECTORIES, PATHS & FILENAMES
# ------------------------------
PROC_DATA_DIR = '/lustre/ific.uv.es/prj/gl/neutrinos/users/ccortesp/NEXT-100/Th_analysis/h5/runs/'
OUTPUT_DIR  = '/lustre/ific.uv.es/prj/gl/neutrinos/users/ccortesp/NEXT-100/Th_analysis/h5/'
SUMMARY_DIR = '/lhome/ific/c/ccortesp/Analysis/NEXT-100/Th_analysis/txt/summaries/'

RUNS_INFO_PATH = os.path.join('/lhome/ific/c/ccortesp/Analysis/NEXT-100/Th_analysis/utilities/runs_information.csv')

SUMMARY_FILENAME = 'summary_LDC_' + FILE_TAG +'.csv'
SUMMARY_PATH = os.path.join(SUMMARY_DIR, SUMMARY_FILENAME)

# -----------------------------
# ALPHA/ELECTRON DISCRIMINATION
# -----------------------------
SIZE_THRESHOLD = 2e3          # in [# of hits]
S1_ENERGY_THRESHOLD = 900     # in [PE]

# ----------------
# DETECTOR REGIONS
# ----------------
# Geometric boundaries for event classification.
Z_LOW = 40          # in [mm]
Z_UP  = 1147        # in [mm]
R_UP  = 451.65      # in [mm]

# -------------
# FINAL COLUMNS
# -------------
INFO_COLS = ['event', 'global_event', 'time']
DATA_COLS = ['run_number', 'particle', 'region']
EVT_COLS = ['nS1', 'nS2', 'n_cluster', 'old_n_hits', 'n_hits', 'E_evt']
POS_COLS = ['X_bary', 'Y_bary', 'Z_bary', 'X_min', 'X_max', 'Y_min', 'Y_max', 'Z_min', 'Z_max', 'R_max']
S1_PULSE_COLS = ['S1e', 'S1e_corr', 'S1w', 'S1h', 'S1t']
S2_PULSE_COLS = ['S2e', 'S2w', 'S2h', 'S2t', 'S2q']

FINAL_COLS = INFO_COLS + DATA_COLS + EVT_COLS + POS_COLS + S1_PULSE_COLS + S2_PULSE_COLS  

### Runs & Summary Information

In [4]:
# # Runs information
# RUNS_INFO_DF = pd.read_csv(RUNS_INFO_PATH)
# RUNS_INFO_DF.columns = RUNS_INFO_DF.columns.str.strip()
# RUNS_INFO_DF

In [4]:
# Summary of the processed runs
summary_LDC_df = pd.read_csv(SUMMARY_PATH)
summary_LDC_df.drop(columns=['Unnamed: 0'], inplace=True)
summary_LDC_df.columns = summary_LDC_df.columns.str.strip()
summary_LDC_df.sort_values(by='Run_ID', inplace=True)
summary_LDC_df

,Run_ID,LDC,n_Files,Sophronia,Clean,Z_Positive,S1_Cut
0,15589,1,all,185551,185550,143961,113229
1,15589,2,all,185152,185152,143959,113090
2,15589,3,all,185280,185280,143869,113488
3,15589,4,all,185089,185089,143536,112979
4,15589,5,all,185515,185514,143822,113469
5,15589,6,all,185215,185215,143579,113006
6,15589,7,all,185187,185185,143605,113021


In [5]:
SUMMARY_DF = summary_LDC_df.drop(columns=['LDC', 'n_Files']).groupby('Run_ID').sum().reset_index()
SUMMARY_DF

,Run_ID,Sophronia,Clean,Z_Positive,S1_Cut
0,15589,1296989,1296985,1006331,792282


# Merge by Production Condition

- For data, the production are differenciated by _data period_ and _detector condition_

### Data

In [7]:
# # Select runs to use according to the notebook configuration
# if DATA_PERIOD is not None:
#     runs_to_analyze = RUNS_INFO_DF.loc[RUNS_INFO_DF['period'] == DATA_PERIOD, 'run_number'].values
#     if DETECTOR_CONDITION is not None:
#         runs_to_analyze = RUNS_INFO_DF.loc[(RUNS_INFO_DF['period'] == DATA_PERIOD) & (RUNS_INFO_DF['condition'] == DETECTOR_CONDITION), 'run_number'].values

# # Selection
# print(f"\nSelected {len(runs_to_analyze)} runs for merge:")
# print(runs_to_analyze)

In [6]:
runs_to_analyze = [15589]

In [7]:
total_corr_time = 0
# total_ok = 0
# total_lost = 0
total_processed_events = 0
all_processed_df = []

for run_id in runs_to_analyze:

    print(f"--- Merging Run {run_id} ---")
    if run_id not in SUMMARY_DF['Run_ID'].values:
        print(f"  → Run {run_id} not found in summary file. Skipping...")
        continue

    # # --- Run Information --- #
    # # Extract run information from the summary dataframe
    # run_duration = SUMMARY_DF.loc[SUMMARY_DF['Run_ID'] == run_id, 'Duration'].values[0]
    # run_OK   = SUMMARY_DF.loc[SUMMARY_DF['Run_ID'] == run_id, 'OK'].values[0]
    # run_LOST = SUMMARY_DF.loc[SUMMARY_DF['Run_ID'] == run_id, 'LOST'].values[0]
    # # Calculate DAQ efficiency and corrected time
    # DAQe_CV, DAQe_error = crudo.ff.efficiency(run_OK, run_LOST)
    # run_corr_time    = run_duration * DAQe_CV
    # total_corr_time += run_corr_time
    # Accumulate processed events
    processed_events = SUMMARY_DF.loc[SUMMARY_DF['Run_ID'] == run_id, 'S1_Cut'].values[0]
    total_processed_events += processed_events

    # --- Dataframes --- #
    files_to_merge = sorted(glob.glob(os.path.join(PROC_DATA_DIR, f"processed_run_{run_id}*{FILE_TAG}*.h5")))
    for run_file in files_to_merge:
        run_df = pd.read_hdf(run_file, key='Events')
        run_df['run_number'] = run_id
        all_processed_df.append(run_df)

# --- Print Summary --- #
# print(f"\nFor period {DATA_PERIOD} with condition '{DETECTOR_CONDITION}':\n  Corrected Time = {total_corr_time:.4f} s")
# Concatenate all dataframes
MERGED_DF = pd.concat(all_processed_df, ignore_index=True)
print(f"Dataframe merged successfully:\n  Total processed events: {total_processed_events}")

--- Merging Run 15589 ---
Dataframe merged successfully:
  Total processed events: 792282


### Compute Global Event ID

In [9]:
COMP_COLS = ['event', 'run_number']

# An original event is defined as a row in dataframe where at least one of the columns 
# ('event', 'run_number') differs from the corresponding row below it (using `shift`).
event_OG = (MERGED_DF[COMP_COLS] != MERGED_DF[COMP_COLS].shift())

# If any column in event_OG is True, it means the row corresponds to the start of a new original event block.
new_event_block = event_OG.any(axis=1)

# Use `cumsum()` on the boolean mask to create a unique identifier for each contiguous block of hits 
# that belong to the same original event.
unique_block_id = new_event_block.cumsum()

# Assign a unique global event ID to each block of original events.
# The `factorize` function generates a unique integer code for each unique block ID.
MERGED_DF['global_event'] = pd.factorize(unique_block_id)[0]
print(f"{MERGED_DF['global_event'].nunique()} unique global events identified.")

792282 unique global events identified.


In [10]:
MERGED_DF

,event,nS1,nS2,old_n_hits,time,S1e,S1e_corr,S1w,S1h,S1t,...,Z_bary,X_min,X_max,Y_min,Y_max,Z_min,Z_max,R_max,run_number,global_event
0,8,1,2,929,1.751990e+09,151.747345,202.112048,375.0,24.432436,752125.0,...,727.308837,-373.775,443.375,29.225,337.725,546.986210,1212.989933,502.019787,15589,0
1,8,1,2,929,1.751990e+09,151.747345,149.495966,375.0,24.432436,752125.0,...,727.308837,-373.775,443.375,29.225,337.725,546.986210,1212.989933,502.019787,15589,0
2,29,1,1,1448,1.751990e+09,391.903259,488.807104,650.0,70.675652,604150.0,...,720.406654,-3.575,227.675,-479.925,-247.675,652.442036,835.121603,487.977096,15589,1
3,71,1,4,706,1.751990e+09,130.162003,195.454648,350.0,22.418476,981275.0,...,550.958364,-265.925,149.925,183.725,353.775,355.748117,1340.952626,390.691990,15589,2
4,71,1,4,706,1.751990e+09,130.162003,147.733884,350.0,22.418476,981275.0,...,550.958364,-265.925,149.925,183.725,353.775,355.748117,1340.952626,390.691990,15589,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1445060,3847158,1,6,1935,1.752077e+09,374.241272,398.025118,775.0,64.607521,220650.0,...,1096.086858,-373.775,490.025,-324.925,383.875,194.291083,2122.602524,491.099915,15589,792280
1445061,3847158,1,6,1935,1.752077e+09,374.241272,346.866498,775.0,64.607521,220650.0,...,1096.086858,-373.775,490.025,-324.925,383.875,194.291083,2122.602524,491.099915,15589,792280
1445062,3847158,1,6,1935,1.752077e+09,374.241272,267.602582,775.0,64.607521,220650.0,...,1096.086858,-373.775,490.025,-324.925,383.875,194.291083,2122.602524,491.099915,15589,792280
1445063,3847179,1,2,278,1.752077e+09,42.227760,65.447167,425.0,6.814364,1031225.0,...,391.284386,-281.475,196.575,137.075,276.025,316.092193,691.413314,373.445587,15589,792281


# Tagging Events

### By Particle

In [10]:
particle_tagged_MERGED_DF = crudo.dm.tag_particles( MERGED_DF
                                                  , size_threshold=SIZE_THRESHOLD
                                                  , s1_energy_threshold=S1_ENERGY_THRESHOLD
                                                  , event_column='global_event' )
particle_tagged_MERGED_DF

,event,nS1,nS2,old_n_hits,time,S1e,S1e_corr,S1w,S1h,S1t,...,X_min,X_max,Y_min,Y_max,Z_min,Z_max,R_max,run_number,global_event,particle
0,8,1,2,929,1.751990e+09,151.747345,202.112048,375.0,24.432436,752125.0,...,-373.775,443.375,29.225,337.725,546.986210,1212.989933,502.019787,15589,0,electron
1,8,1,2,929,1.751990e+09,151.747345,149.495966,375.0,24.432436,752125.0,...,-373.775,443.375,29.225,337.725,546.986210,1212.989933,502.019787,15589,0,electron
2,29,1,1,1448,1.751990e+09,391.903259,488.807104,650.0,70.675652,604150.0,...,-3.575,227.675,-479.925,-247.675,652.442036,835.121603,487.977096,15589,1,electron
3,71,1,4,706,1.751990e+09,130.162003,195.454648,350.0,22.418476,981275.0,...,-265.925,149.925,183.725,353.775,355.748117,1340.952626,390.691990,15589,2,electron
4,71,1,4,706,1.751990e+09,130.162003,147.733884,350.0,22.418476,981275.0,...,-265.925,149.925,183.725,353.775,355.748117,1340.952626,390.691990,15589,2,electron
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1445060,3847158,1,6,1935,1.752077e+09,374.241272,398.025118,775.0,64.607521,220650.0,...,-373.775,490.025,-324.925,383.875,194.291083,2122.602524,491.099915,15589,792280,electron
1445061,3847158,1,6,1935,1.752077e+09,374.241272,346.866498,775.0,64.607521,220650.0,...,-373.775,490.025,-324.925,383.875,194.291083,2122.602524,491.099915,15589,792280,electron
1445062,3847158,1,6,1935,1.752077e+09,374.241272,267.602582,775.0,64.607521,220650.0,...,-373.775,490.025,-324.925,383.875,194.291083,2122.602524,491.099915,15589,792280,electron
1445063,3847179,1,2,278,1.752077e+09,42.227760,65.447167,425.0,6.814364,1031225.0,...,-281.475,196.575,137.075,276.025,316.092193,691.413314,373.445587,15589,792281,electron


### By Detector Region

In [ ]:
# region_tagged_MERGED_DF = crudo.dm.tag_event_by_detector_region( particle_tagged_MERGED_DF
#                                                                , z_cut_low=Z_LOW
#                                                                , z_cut_high=Z_UP
#                                                                , r_cut_high=R_UP
#                                                                , event_column='global_event' )


region_tagged_MERGED_DF

,event,nS1,nS2,old_n_hits,time,S1e,S1e_corr,S1w,S1h,S1t,...,X_max,Y_min,Y_max,Z_min,Z_max,R_max,run_number,global_event,particle,region
0,8,1,2,929,1.751990e+09,151.747345,202.112048,375.0,24.432436,752125.0,...,443.375,29.225,337.725,546.986210,1212.989933,502.019787,15589,0,electron,unclassified
1,8,1,2,929,1.751990e+09,151.747345,149.495966,375.0,24.432436,752125.0,...,443.375,29.225,337.725,546.986210,1212.989933,502.019787,15589,0,electron,unclassified
2,29,1,1,1448,1.751990e+09,391.903259,488.807104,650.0,70.675652,604150.0,...,227.675,-479.925,-247.675,652.442036,835.121603,487.977096,15589,1,electron,unclassified
3,71,1,4,706,1.751990e+09,130.162003,195.454648,350.0,22.418476,981275.0,...,149.925,183.725,353.775,355.748117,1340.952626,390.691990,15589,2,electron,fiducial
4,71,1,4,706,1.751990e+09,130.162003,147.733884,350.0,22.418476,981275.0,...,149.925,183.725,353.775,355.748117,1340.952626,390.691990,15589,2,electron,fiducial
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1445060,3847158,1,6,1935,1.752077e+09,374.241272,398.025118,775.0,64.607521,220650.0,...,490.025,-324.925,383.875,194.291083,2122.602524,491.099915,15589,792280,electron,unclassified
1445061,3847158,1,6,1935,1.752077e+09,374.241272,346.866498,775.0,64.607521,220650.0,...,490.025,-324.925,383.875,194.291083,2122.602524,491.099915,15589,792280,electron,unclassified
1445062,3847158,1,6,1935,1.752077e+09,374.241272,267.602582,775.0,64.607521,220650.0,...,490.025,-324.925,383.875,194.291083,2122.602524,491.099915,15589,792280,electron,unclassified
1445063,3847179,1,2,278,1.752077e+09,42.227760,65.447167,425.0,6.814364,1031225.0,...,196.575,137.075,276.025,316.092193,691.413314,373.445587,15589,792281,electron,fiducial


# Output

In [12]:
FINAL_DF = region_tagged_MERGED_DF[FINAL_COLS].copy()
FINAL_DF

,event,global_event,time,run_number,particle,region,nS1,nS2,n_cluster,old_n_hits,...,S1e,S1e_corr,S1w,S1h,S1t,S2e,S2w,S2h,S2t,S2q
0,8,0,1.751990e+09,15589,electron,unclassified,1,2,5,929,...,151.747345,202.112048,375.0,24.432436,752125.0,150070.453125,281.050,3798.209229,1417489.625,68422.437500
1,8,0,1.751990e+09,15589,electron,unclassified,1,2,5,929,...,151.747345,149.495966,375.0,24.432436,752125.0,45446.609375,238.050,3060.650635,2143500.500,38754.382812
2,29,1,1.751990e+09,15589,electron,unclassified,1,1,1,1448,...,391.903259,488.807104,650.0,70.675652,604150.0,284933.000000,510.125,4855.565430,1409485.875,118341.734375
3,71,2,1.751990e+09,15589,electron,fiducial,1,4,4,706,...,130.162003,195.454648,350.0,22.418476,981275.0,142723.734375,117.650,6206.925781,1413484.625,36185.945312
4,71,2,1.751990e+09,15589,electron,fiducial,1,4,4,706,...,130.162003,147.733884,350.0,22.418476,981275.0,32346.220703,149.925,2346.959473,2004489.000,22086.957031
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1445060,3847158,792280,1.752077e+09,15589,electron,unclassified,1,6,9,1935,...,374.241272,398.025118,775.0,64.607521,220650.0,257238.359375,206.275,4704.867188,1406478.750,70888.632812
1445061,3847158,792280,1.752077e+09,15589,electron,unclassified,1,6,9,1935,...,374.241272,346.866498,775.0,64.607521,220650.0,33476.074219,88.300,1930.657837,1787477.000,12858.297852
1445062,3847158,792280,1.752077e+09,15589,electron,unclassified,1,6,9,1935,...,374.241272,267.602582,775.0,64.607521,220650.0,26483.140625,93.350,2331.652100,2665487.500,11706.981445
1445063,3847179,792281,1.752077e+09,15589,electron,fiducial,1,2,2,278,...,42.227760,65.447167,425.0,6.814364,1031225.0,57837.691406,257.525,4834.141602,1406492.375,44652.847656


In [ ]:
# H5 output filename
merged_filename = 'merged_tagged_runs_' + FILE_TAG + '.h5'
    
merged_path = os.path.join(OUTPUT_DIR, merged_filename)
print(f"\nSaving merged dataframe to: {merged_path}")


Saving merged dataframe to: /lustre/ific.uv.es/prj/gl/neutrinos/users/ccortesp/NEXT-100/Th_analysis/h5/merged_tagged_runs_LPR_p2_v2_radial.h5


In [15]:
FINAL_DF.to_hdf(merged_path, key='Events', mode='w', format='table')
print('Done!')

Done!


Update summary file per run!

In [18]:
output_filename = os.path.join(SUMMARY_DIR, f'summary_{FILE_TAG}_processed.csv')
SUMMARY_DF.to_csv(output_filename)
print('Done! Summary updated in:', output_filename)

Done! Summary updated in: /lhome/ific/c/ccortesp/Analysis/NEXT-100/Th_analysis/txt/summaries/summary_LPR_p2_v2_processed.csv
